# Energinet (Denmark) – Simple Bids

This notebook covers simple bid submission to Energinet, the Danish TSO.  Denmark
has several important differences from the other Nordic TSOs:

| Parameter | Value |
|---|---|
| Receiver EIC | `10X1001A1001A248` |
| Control area EIC | `10Y1001A1001A796` |
| Minimum bid | 1 MW |
| Heartbeat | Not implemented at go-live |
| BRP/BSP model | BRP acts as BSP — always use role A46 (BSP) |
| Resource coding scheme | A01 (EIC) |
| Production type | **Mandatory** — `mktPSRType.psrType` (DK-specific schema) |
| Note field | Optional free text, passed through to activation document |
| Direct activation | Local DA model (separate bid pairs, added v1.2.0) |
| Conditional links | A71 and A72 not supported |

## What this notebook covers

1. Simple divisible bid with production type
2. All four production types
3. Validation showing `psr_type` is required
4. XML output with Denmark-specific fields
5. Local DA model: two technically linked bids across MTUs

## Imports

In [ ]:
from nexa_mfrr_eam import (
    Bid,
    BidDocument,
    BiddingZone,
    MARIMode,
    MarketProductType,
    ProductionType,
    TechnicalLink,
    TSO,
    Direction,
)

# Synthetic Energinet BSP/BRP party ID (EIC coding scheme)
BSP_PARTY_ID = "10XBRP-001-----A"
BSP_CODING_SCHEME = "A01"

# Synthetic resource objects (EIC)
WIND_ONSHORE_EIC = "10XWIND-ON-DK1--W"
WIND_OFFSHORE_EIC = "10XWIND-OFF-DK1-W"
SOLAR_EIC = "10XSOLAR-DK1----W"

## Part 1: Simple Divisible Bid With Production Type

Energinet requires `mktPSRType.psrType` on every bid time series.  This is a
**Denmark-specific schema extension** not present in the standard NBM XSD.  The
library serializes it as a `<mktPSRType.psrType>` element after the standard
bid attributes.

The four supported production types are:

| Code | `ProductionType` | Technology |
|------|-----------------|------------|
| B16  | `SOLAR`          | Solar power |
| B18  | `WIND_OFFSHORE`  | Wind offshore |
| B19  | `WIND_ONSHORE`   | Wind onshore |
| B20  | `OTHER`          | Other technology |

Use `bid.model_copy(update={"psr_type": ProductionType.X.value})` to attach the
production type to a bid built with `SimpleBidBuilder`.

> **Note:** A future release will expose `.production_type()` directly on the builder.

In [ ]:
# Build a simple divisible up-regulation bid for a wind onshore resource
raw_bid = (
    Bid.up(volume_mw=25, price_eur=72.50)
    .divisible(min_volume_mw=5)
    .for_mtu("2026-03-21T10:00Z")
    .resource(WIND_ONSHORE_EIC, coding_scheme="A01")
    .bidding_zone(BiddingZone.DK1)
    .product_type(MarketProductType.SCHEDULED_ONLY)  # A05 — no direct activation
    .build()
)

# Attach the mandatory production type (B19 = wind onshore)
bid = raw_bid.model_copy(update={"psr_type": ProductionType.WIND_ONSHORE.value})

print(f"Bid mRID:       {bid.mrid}")
print(f"Direction:      {bid.flow_direction}")
print(f"Volume:         {bid.period.point.quantity} MW")
print(f"Min volume:     {bid.period.point.minimum_quantity} MW")
print(f"Price:          {bid.period.point.energy_price} EUR/MWh")
print(f"MTU:            {bid.period.time_interval_start.strftime('%Y-%m-%dT%H:%MZ')}")
print(f"Production type: {bid.psr_type} (B19 = wind onshore)")

## Part 2: Validation – psr_type is required for Energinet

If you submit a bid to `TSO.ENERGINET` without `psr_type`, the document-level
validation returns an error describing the missing field.

In [ ]:
# Build a bid WITHOUT production type
bid_no_psr = (
    Bid.up(volume_mw=20, price_eur=65.00)
    .divisible(min_volume_mw=5)
    .for_mtu("2026-03-21T10:00Z")
    .resource(WIND_ONSHORE_EIC, coding_scheme="A01")
    .bidding_zone(BiddingZone.DK1)
    .product_type(MarketProductType.SCHEDULED_ONLY)
    .build()
)

doc_missing_psr = (
    BidDocument(tso=TSO.ENERGINET)
    .sender(party_id=BSP_PARTY_ID, coding_scheme=BSP_CODING_SCHEME)
    .add_bid(bid_no_psr)
    .build()
)

errors = doc_missing_psr.validate(mari_mode=MARIMode.PRE_MARI)
print("Validation errors (expected):")
for e in errors:
    print(f"  {e}")

## Part 3: All Four Production Types

Here we build one bid per production type and confirm they all pass validation.

In [ ]:
production_types = [
    (ProductionType.SOLAR, "10XSOLAR-DK1----W", BiddingZone.DK1),
    (ProductionType.WIND_OFFSHORE, "10XWOFF-DK1-----W", BiddingZone.DK1),
    (ProductionType.WIND_ONSHORE, WIND_ONSHORE_EIC, BiddingZone.DK1),
    (ProductionType.OTHER, "10XOTHER-DK1----W", BiddingZone.DK2),
]

bids_with_types = []
for ptype, resource_eic, zone in production_types:
    raw = (
        Bid.up(volume_mw=10, price_eur=80.0)
        .divisible(min_volume_mw=1)
        .for_mtu("2026-03-21T10:00Z")
        .resource(resource_eic, coding_scheme="A01")
        .bidding_zone(zone)
        .product_type(MarketProductType.SCHEDULED_ONLY)
        .build()
    )
    b = raw.model_copy(update={"psr_type": ptype.value})
    bids_with_types.append(b)
    print(f"  {ptype.name:15s} ({ptype.value})  → {zone.name}")

doc_multi = (
    BidDocument(tso=TSO.ENERGINET)
    .sender(party_id=BSP_PARTY_ID, coding_scheme=BSP_CODING_SCHEME)
    .add_bids(bids_with_types)
    .build()
)

errors = doc_multi.validate(mari_mode=MARIMode.PRE_MARI)
if errors:
    for e in errors:
        print(f"ERROR: {e}")
else:
    print(f"\nDocument valid: {doc_multi.time_series_count} bids, no errors")

## Part 4: XML Output With Denmark-Specific Fields

The serializer emits `<mktPSRType.psrType>` and (optionally) `<Note>` after the
standard XSD elements.  Their position in the Denmark-specific schema will be
confirmed once the DK XSD is obtained from Energinet.

In [ ]:
# Bid with both psr_type and a Note
bid_with_note = raw_bid.model_copy(
    update={
        "psr_type": ProductionType.WIND_ONSHORE.value,
        "note": "Custom BRP reference: PROJ-2026-0042",
    }
)

doc_note = (
    BidDocument(tso=TSO.ENERGINET)
    .sender(party_id=BSP_PARTY_ID, coding_scheme=BSP_CODING_SCHEME)
    .add_bid(bid_with_note)
    .build()
)

xml_output = doc_note.to_xml().decode("utf-8")

# Show only the Bid_TimeSeries section
start = xml_output.index("<Bid_TimeSeries>")
end = xml_output.index("</Bid_TimeSeries>") + len("</Bid_TimeSeries>")
print(xml_output[start:end])

## Part 5: Local DA Model – Two Technically Linked Bids

Denmark uses a **local direct activation model** where the DA bid and its
linked next-quarter bid are two **separate simple bids** with a shared
`linkedBidsIdentification` UUID.  They can have different prices and volumes.

This is different from the other TSOs where DA capability is indicated by
`MarketProductType.SCHEDULED_AND_DIRECT` (A07).  In Denmark, both bids
use the same product type; the link is what signals the DA relationship.

### Example: wind farm with DA capability in DK1

- **QH0 bid (10:00)**: The activatable bid — direct activation starts here.
- **QH+1 bid (10:15)**: Linked continuation — if QH0 is direct-activated and
  the unit is still ramping, this bid covers the next quarter.

In [ ]:
import uuid

# Shared technical link ID
link_id = str(uuid.uuid4())

# QH0 bid – direct activatable
da_bid_raw = (
    Bid.up(volume_mw=30, price_eur=85.00)
    .divisible(min_volume_mw=5)
    .for_mtu("2026-03-21T10:00Z")
    .resource(WIND_ONSHORE_EIC, coding_scheme="A01")
    .bidding_zone(BiddingZone.DK1)
    .product_type(MarketProductType.SCHEDULED_ONLY)
    .technical_link(link_id)
    .build()
)
da_bid = da_bid_raw.model_copy(update={"psr_type": ProductionType.WIND_ONSHORE.value})

# QH+1 bid – continuation (different price, same resource/zone)
next_qh_raw = (
    Bid.up(volume_mw=25, price_eur=78.00)  # slightly lower price for Q+1
    .divisible(min_volume_mw=5)
    .for_mtu("2026-03-21T10:15Z")
    .resource(WIND_ONSHORE_EIC, coding_scheme="A01")
    .bidding_zone(BiddingZone.DK1)
    .product_type(MarketProductType.SCHEDULED_ONLY)
    .technical_link(link_id)  # same link ID
    .build()
)
next_qh_bid = next_qh_raw.model_copy(
    update={"psr_type": ProductionType.WIND_ONSHORE.value}
)

print(f"Link ID: {link_id}")
print(f"QH0  ({da_bid.period.time_interval_start.strftime('%H:%M')}): "
      f"{da_bid.period.point.quantity} MW @ {da_bid.period.point.energy_price} EUR/MWh")
print(f"QH+1 ({next_qh_bid.period.time_interval_start.strftime('%H:%M')}): "
      f"{next_qh_bid.period.point.quantity} MW @ {next_qh_bid.period.point.energy_price} EUR/MWh")
print(f"Shared link: {da_bid.linked_bids_identification == next_qh_bid.linked_bids_identification}")

In [ ]:
doc_da = (
    BidDocument(tso=TSO.ENERGINET)
    .sender(party_id=BSP_PARTY_ID, coding_scheme=BSP_CODING_SCHEME)
    .add_bid(da_bid)
    .add_bid(next_qh_bid)
    .build()
)

errors = doc_da.validate(mari_mode=MARIMode.PRE_MARI)
if errors:
    for e in errors:
        print(f"ERROR: {e}")
else:
    print(f"Document valid: {doc_da.time_series_count} bids")
    print(f"Period: {doc_da.model.reserve_bid_period_start.strftime('%H:%MZ')} "
          f"to {doc_da.model.reserve_bid_period_end.strftime('%H:%MZ')}")

## Summary

| Feature | How to use | XML element |
|---------|-----------|-------------|
| Production type (mandatory) | `.model_copy(update={"psr_type": ProductionType.X.value})` | `<mktPSRType.psrType>` |
| Note (optional) | `.model_copy(update={"note": "text"})` | `<Note>` |
| Local DA model | `.technical_link(shared_uuid)` on two consecutive-MTU bids | `<linkedBidsIdentification>` |
| TSO | `BidDocument(tso=TSO.ENERGINET)` | Receiver EIC `10X1001A1001A248` |

**What is NOT supported for Energinet:**
- Inclusive bid groups
- Period shift
- Heartbeat
- Conditional link types A71 and A72
- Changing product type on update (must cancel + resubmit)